# H2O — Harness-Integrated Implementation

This is H2O built directly from the shared baseline harness
(`_baseline_template.ipynb`), with **only Block 7 changed** to implement
H2O's actual heavy-hitter eviction algorithm — every other block (env setup,
shared config, dataset loaders, scoring/generation/evaluation, results
table) is byte-identical to what `KVQuant_Implementation_v2.ipynb` uses, so
the two methods' results are directly comparable: same model, same seed,
same `N_EVAL_SAMPLES`/`MAX_LENGTH`/GSM8K prompt/generation budget, same
dataset sources and splits.

H2O evicts KV-cache tokens at every decoding step, keeping only a window of
the most **recent** tokens plus the top-K **heavy-hitter** tokens (ranked by
cumulative attention score), discarding everything else. Because eviction
physically shrinks the cache tensors, this method's memory number is a
genuine **measurement** (`kv_cache_accounting == "measured"`) — unlike
KVQuant's simulated quantization, which never physically shrinks anything
and so can only report a **calculated** formula-based number. Both are
reported through the same `reported_kv_cache_gb` column, tagged with which
kind of number they are, so they're never silently compared as if they were
the same kind of quantity.

**Not aligned with KVQuant, by design** (structural properties of the two
methods, not experimental-design bugs):
- **Attention backend**: H2O must use eager attention (it needs real
  attention weights for heavy-hitter scoring); KVQuant auto-selects
  flash/sdpa. Set `FORCE_ATTN_IMPL = "sdpa"` (or `"eager"`) in the KVQuant
  notebook's Block 3 if you want a controlled confound check that isolates
  the attention-backend effect from the compression-method effect.
- **Calibration**: KVQuant calibrates once (Fisher information on
  WikiText2) and is evaluated cross-domain; H2O is calibration-free and
  adapts its eviction policy per-sequence, online, during generation. This
  is a real property of the two methods, not something to force into
  alignment.

In [ ]:
# Block 1 - Environment setup
# Run once per fresh runtime. Package versions are pinned so environment
# differences are never a confound between compression methods tested with
# this template.

from google.colab import drive
drive.mount("/content/drive")

!python -m pip install -q --no-deps \
  "transformers==4.43.4" \
  "accelerate==0.33.0" \
  "tokenizers==0.19.1" \
  "huggingface_hub==0.36.2" \
  sentencepiece \
  einops

!python -m pip install -q \
  "datasets==2.14.5" \
  tqdm \
  matplotlib

!python -m pip install -q --no-deps --force-reinstall "huggingface_hub==0.36.2"

# Optional: install a FlashAttention wheel from Drive if you have one built.
import os, sys, glob, subprocess

wheel_candidates = glob.glob("/content/drive/MyDrive/flash_attn-*.whl") + glob.glob("/content/flash_attn-*.whl")
if wheel_candidates:
    FLASH_ATTN_WHEEL = sorted(wheel_candidates)[-1]
    print("Installing FlashAttention wheel:", FLASH_ATTN_WHEEL)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-deps", "--force-reinstall", FLASH_ATTN_WHEEL])
else:
    print("No FlashAttention wheel found; will fall back to sdpa/eager as needed.")

# Optional HuggingFace login for gated models. Never hardcode a token here --
# pull it from Colab's secret manager or an environment variable instead.
try:
    from google.colab import userdata
    _hf_token = userdata.get("HF_TOKEN")
except Exception:
    _hf_token = os.environ.get("HF_TOKEN")

if _hf_token:
    from huggingface_hub import login
    login(token=_hf_token)
    print("Logged in to HuggingFace")
else:
    print("No HF_TOKEN found -- skipping login (fine for public models like TinyLlama)")

print("Block 1 finished. Now run Block 2.")

In [ ]:
# Block 2 - Imports, GPU check, attention backend detection

import gc
import math
import time
import random
import os

import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt

import datasets
import transformers
import huggingface_hub
from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("torch:", torch.__version__)
print("cuda:", torch.version.cuda)
print("datasets:", datasets.__version__)
print("transformers:", transformers.__version__)
print("huggingface_hub:", huggingface_hub.__version__)
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO CUDA")

HAS_CUDA = torch.cuda.is_available()
DEVICE = torch.device("cuda" if HAS_CUDA else "cpu")
MODEL_DTYPE = torch.float16 if HAS_CUDA else torch.float32


def sync_if_cuda():
    if HAS_CUDA:
        torch.cuda.synchronize()


def clear_memory():
    gc.collect()
    if HAS_CUDA:
        torch.cuda.empty_cache()


FLASH_AVAILABLE = False
FLASH_IMPORT_ERROR = None

if HAS_CUDA:
    try:
        import flash_attn
        from flash_attn import flash_attn_func
        FLASH_AVAILABLE = True
    except Exception as e:
        FLASH_IMPORT_ERROR = repr(e)

if HAS_CUDA and FLASH_AVAILABLE:
    FAST_ATTN_IMPL = "flash_attention_2"
elif HAS_CUDA:
    FAST_ATTN_IMPL = "sdpa"
else:
    FAST_ATTN_IMPL = "eager"

print()
print("Device:", DEVICE)
print("Model dtype:", MODEL_DTYPE)
print("FlashAttention available:", FLASH_AVAILABLE)
if FLASH_IMPORT_ERROR is not None:
    print("FlashAttention import error:", FLASH_IMPORT_ERROR)
print("Fastest auto-selected attention backend:", FAST_ATTN_IMPL)

if not HAS_CUDA:
    print("WARNING: No GPU detected. This will be slow.")

## Shared experiment configuration

Copy these values identically into every method's notebook. Only
`METHOD_NAME` and Block 7's `REQUIRED_ATTN_IMPL`/hooks should differ between
copies — that is what keeps the comparison across methods fair.

In [ ]:
# Block 3 - Experiment settings.

LOCAL_MODEL_PATH = "/content/tinyllama-1.1b"
HF_MODEL_ID = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
MODEL_ID = LOCAL_MODEL_PATH if os.path.exists(LOCAL_MODEL_PATH) else HF_MODEL_ID

SHARED_SEED = 0
N_EVAL_SAMPLES = 64        # PTB/WikiText-103 window count AND GSM8K question count
MAX_LENGTH = 1024          # window length for PTB/WikiText-103 (a.k.a. seqlen)
GSM8K_MAX_NEW_TOKENS = 256
USE_GSM8K_FEWSHOT = True
TBT_GENERATED_TOKENS = 20  # decode length used for TTFT/TBT probes

# Set per method (also set again in Block 7 -- keep both in sync).
METHOD_NAME = "h2o"

random.seed(SHARED_SEED)
torch.manual_seed(SHARED_SEED)

GSM8K_FEWSHOT_PREFIX = (
    "You are solving grade-school math word problems.\n"
    "Show the calculation briefly, then end with exactly this format:\n"
    "#### <final number>\n\n"
    "Question: Mia has 3 marbles and buys 4 more. How many marbles does she have?\n"
    "Answer: Mia has 3 + 4 = 7 marbles.\n"
    "#### 7\n\n"
    "Question: A box has 6 pencils. Sam buys 5 boxes. How many pencils does Sam buy?\n"
    "Answer: Sam buys 6 * 5 = 30 pencils.\n"
    "#### 30\n"
)

print("Model:", MODEL_ID)
print("Method:", METHOD_NAME)
print("MAX_LENGTH / seqlen:", MAX_LENGTH)
print("N_EVAL_SAMPLES:", N_EVAL_SAMPLES)
print("GSM8K max new tokens:", GSM8K_MAX_NEW_TOKENS)
print("GSM8K few-shot prompt:", USE_GSM8K_FEWSHOT)
print("TBT generated tokens:", TBT_GENERATED_TOKENS)

## Dataset suite

| Dataset | Source | Split | Segmentation |
|---|---|---|---|
| PTB | `ptb_text_only` / `penn_treebank` | test | N_EVAL_SAMPLES fixed 512-token windows (perplexity/accuracy); real sentences, token-truncated (TTFT/TBT) |
| WikiText-103 | `wikitext` / `wikitext-103-raw-v1` | test | N_EVAL_SAMPLES fixed 512-token windows (perplexity/accuracy); real paragraphs, token-truncated (TTFT/TBT) |
| GSM8K | `gsm8k` / `main` | test | N_EVAL_SAMPLES questions, seeded shuffle; TTFT/TBT measured on the same few-shot prompt used for accuracy |

In [ ]:
# Block 4 - Tokenizer and generic model loader
# load_model(attn_impl) is shared infrastructure -- every method's Block 7
# adapter calls this instead of writing its own from_pretrained call, so a
# stray kwarg difference can never become a hidden confound between methods.

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=False, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = None
device = DEVICE
active_attn_impl = None


def clear_model():
    global model, device, active_attn_impl
    if model is not None:
        del model
    model = None
    device = DEVICE
    active_attn_impl = None
    clear_memory()


def load_model(attn_impl):
    """Load MODEL_ID with the given attention implementation. Reused as-is by
    every method -- only `attn_impl` should ever differ (set via
    REQUIRED_ATTN_IMPL in Block 7)."""
    global model, device, active_attn_impl

    if model is not None and active_attn_impl == attn_impl:
        return model

    clear_model()

    print(f"Loading model with attn_implementation={attn_impl!r}...")
    model_kwargs = {
        "torch_dtype": MODEL_DTYPE,
        "low_cpu_mem_usage": True,
        "attn_implementation": attn_impl,
        "trust_remote_code": True,
    }
    if HAS_CUDA:
        model_kwargs["device_map"] = {"": 0}

    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **model_kwargs)
    active_attn_impl = attn_impl

    if not HAS_CUDA:
        model = model.to(DEVICE)

    model.eval()
    model.config.use_cache = True
    device = next(model.parameters()).device

    print("Loaded model on:", device, "| dtype:", next(model.parameters()).dtype, "| attn:", active_attn_impl)
    return model

In [ ]:
# Block 5 - Dataset loaders: PTB, WikiText-103, GSM8K
#
# Fixes applied here relative to the earlier H2O/KVQuant notebooks (found
# while diffing them for a baseline comparison):
#   - Natural-prompt truncation is token-based everywhere (tokenizer(...)[:,
#     :max_tokens]), not a character heuristic (`text[:max_tokens*6]`) --
#     so "capped at 256 tokens" means the same thing on every dataset.
#   - GSM8K has ONE canonical source ("gsm8k"/"main"), so the seeded shuffle
#     always draws from the same row order regardless of which method's copy
#     of this notebook is running.
#   - GSM8K's TTFT/TBT is measured on the SAME fewshot-prefixed prompt used
#     for accuracy (see Block 9) -- not a separate, shorter, un-prefixed
#     "natural prompt" -- so latency numbers reflect the prompt length the
#     model actually has to prefill for the real task.

import re


def build_fixed_windows(ids, window_len, n_windows):
    total_needed = window_len * n_windows
    ids = ids[:total_needed]
    if len(ids) < window_len:
        raise ValueError(f"Corpus too short for even one window of {window_len} tokens (got {len(ids)} tokens).")
    n_actual = len(ids) // window_len
    return [ids[i * window_len:(i + 1) * window_len] for i in range(n_actual)]


def build_natural_prompts(texts, n_eval, seed, tok, min_tokens=3, max_tokens=256):
    """Real, individual, variable-length prompts, token-truncated (not a
    character heuristic), capped to n_eval via deterministic seeded shuffle."""
    candidates = [str(t).strip() for t in texts if str(t).strip()]
    random.Random(seed).shuffle(candidates)
    prompts = []
    for t in candidates:
        ids = tok(t, add_special_tokens=True)["input_ids"]
        if len(ids) >= min_tokens:
            ids = ids[:max_tokens]
            prompts.append(tok.decode(ids, skip_special_tokens=True))
        if len(prompts) >= n_eval:
            break
    return prompts


def safe_load(name, loader_fn):
    try:
        data = loader_fn()
        if isinstance(data, dict) and "windows" in data:
            n_natural = len(data.get("natural_prompts", []))
            print(f"Loaded {name}: {len(data['windows'])} windows, {n_natural} natural prompts")
        else:
            print(f"Loaded {name}: {len(data)} samples")
        return data
    except Exception as e:
        print(f"Skipping {name}; loading failed with: {type(e).__name__}: {e}")
        return None


# -------------------------
# PTB / WikiText-103
# -------------------------

def load_ptb_data(n_eval, seqlen, seed, tok):
    # ptb_text_only is a script-based HF dataset; datasets library versions
    # have been dropping script support (and version combos that still
    # support it clash with trust_remote_code handling for this specific
    # script), so this pulls the raw Penn Treebank test text directly from
    # the standard research mirror used across GPTQ/AWQ-style quantization
    # codebases instead, avoiding datasets.load_dataset entirely for PTB.
    import urllib.request
    url = "https://raw.githubusercontent.com/wojzaremba/lstm/master/data/ptb.test.txt"
    raw_text = urllib.request.urlopen(url).read().decode("utf-8")
    sentences = [s.strip() for s in raw_text.splitlines() if s.strip()]
    full_text = "\n\n".join(sentences)
    ids = tok(full_text, add_special_tokens=False)["input_ids"]
    windows = build_fixed_windows(ids, seqlen, n_eval)
    natural_prompts = build_natural_prompts(sentences, n_eval, seed, tok)
    return {"windows": windows, "natural_prompts": natural_prompts}


def load_wikitext103_data(n_eval, seqlen, seed, tok):
    try:
        ds = load_dataset("wikitext", "wikitext-103-raw-v1", split="test")
    except Exception as e1:
        print("wikitext load failed, trying Salesforce/wikitext. Error was:", type(e1).__name__, e1)
        ds = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1", split="test")

    raw_texts = [str(row["text"]).strip() for row in ds if str(row["text"]).strip()]
    full_text = "\n\n".join(raw_texts)
    ids = tok(full_text, add_special_tokens=False)["input_ids"]
    windows = build_fixed_windows(ids, seqlen, n_eval)
    natural_prompts = build_natural_prompts([t for t in raw_texts if len(t) > 20], n_eval, seed, tok)
    return {"windows": windows, "natural_prompts": natural_prompts}


lm_datasets = {
    "PTB": safe_load("PTB", lambda: load_ptb_data(N_EVAL_SAMPLES, MAX_LENGTH, SHARED_SEED, tokenizer)),
    "WikiText-103": safe_load("WikiText-103", lambda: load_wikitext103_data(N_EVAL_SAMPLES, MAX_LENGTH, SHARED_SEED, tokenizer)),
}
lm_datasets = {k: v for k, v in lm_datasets.items() if v is not None}


# -------------------------
# GSM8K helpers
# -------------------------

def normalize_number_text(text):
    text = str(text).strip().replace(",", "")
    match = re.search(r"-?\d+(?:\.\d+)?", text)
    if not match:
        return ""
    value = match.group(0)
    try:
        f = float(value)
        if abs(f - int(f)) < 1e-9:
            return str(int(f))
    except Exception:
        pass
    return value


def extract_last_number(text):
    text = str(text).replace(",", "")
    matches = re.findall(r"-?\d+(?:\.\d+)?", text)
    return normalize_number_text(matches[-1]) if matches else ""


def extract_gsm8k_final_answer(answer_text):
    answer_text = str(answer_text).replace(",", "")
    if "####" in answer_text:
        extracted = normalize_number_text(answer_text.split("####")[-1])
        if extracted != "":
            return extracted
    patterns = [
        r"(?:final answer|the answer is|answer is|answer)\s*[:=]?\s*(-?\d+(?:\.\d+)?)",
        r"####\s*(-?\d+(?:\.\d+)?)",
    ]
    for pattern in patterns:
        matches = re.findall(pattern, answer_text, flags=re.IGNORECASE)
        if matches:
            return normalize_number_text(matches[-1])
    return extract_last_number(answer_text)


def build_gsm8k_prompt(question):
    if USE_GSM8K_FEWSHOT:
        return GSM8K_FEWSHOT_PREFIX + "\nQuestion: " + str(question).strip() + "\nAnswer:"
    return (
        "Solve this math word problem. Show the calculation briefly and end with exactly: #### <final number>\n\n"
        f"Question: {str(question).strip()}\nAnswer:"
    )


def load_gsm8k_examples(n_eval, seed):
    # Single canonical source so every method's copy of this notebook draws
    # from the same row order for the seeded shuffle below.
    ds = load_dataset("gsm8k", "main", split="test")

    indices = list(range(len(ds)))
    random.Random(seed).shuffle(indices)

    examples = []
    for i in indices:
        row = ds[i]
        final_answer = extract_gsm8k_final_answer(row["answer"])
        prompt = build_gsm8k_prompt(row["question"])
        examples.append({
            "dataset": "GSM8K",
            "prompt": prompt,
            "target": final_answer,
            "answer_type": "number",
            "max_new_tokens": GSM8K_MAX_NEW_TOKENS,
            "question": row["question"],
            "reference_answer": row["answer"],
        })
        if len(examples) >= n_eval:
            break
    return examples


generation_datasets = {"GSM8K": safe_load("GSM8K", lambda: load_gsm8k_examples(N_EVAL_SAMPLES, SHARED_SEED))}
generation_datasets = {k: v for k, v in generation_datasets.items() if v is not None}

print()
print("Language modeling datasets:")
for name, d in lm_datasets.items():
    print(f"  {name}: {len(d['windows'])} windows of {MAX_LENGTH} tokens, {len(d['natural_prompts'])} natural prompts for TTFT/TBT")

print()
print("Generation datasets:")
for name, examples in generation_datasets.items():
    print(f"  {name}: {len(examples)} samples")

## Compression-method plug-in interface

Every method must implement six things (Block 7): `METHOD_NAME`,
`REQUIRED_ATTN_IMPL`, `init_state()`, `make_budgets(total_len)`,
`forward_one_token(...)`, `get_state_cache_tokens(state)`,
`measured_kv_cache_bytes(state)`, `calculated_kv_cache_bytes(state)`.

Everything below Block 7 (scoring, generation, dataset-level evaluation,
result tables) calls only these six things — it never needs to know whether
the method evicts tokens, quantizes them, merges them, or does nothing.

In [ ]:
# Block 6 - Generic KV-cache helpers
# Shared infrastructure -- every method reuses these unmodified. Only Block
# 7's hooks decide HOW the cache is pruned/quantized/reshaped.


def cache_to_legacy(past_key_values):
    if past_key_values is None:
        return None
    if isinstance(past_key_values, tuple):
        return past_key_values
    if hasattr(past_key_values, "to_legacy_cache"):
        return past_key_values.to_legacy_cache()
    raise TypeError(f"Unsupported cache type: {type(past_key_values)}")


def get_cache_tokens(past_key_values):
    past_key_values = cache_to_legacy(past_key_values)
    if past_key_values is None:
        return 0
    return int(past_key_values[0][0].shape[2])


def dense_kv_cache_bytes(past_key_values):
    """Real bytes of whatever tensors are actually resident in
    past_key_values right now, at their current dtype. This is the honest
    'measured' number for methods that physically shrink the cache
    (eviction/merging). For methods that only simulate compression (tensors
    stay full fp16 in memory), this will just equal the dense size -- use
    calculated_kv_cache_bytes (Block 7) to report the compressed size for
    those methods instead, and label it as calculated rather than measured."""
    past_key_values = cache_to_legacy(past_key_values)
    if past_key_values is None:
        return 0
    total = 0
    for key, value in past_key_values:
        total += key.numel() * key.element_size()
        total += value.numel() * value.element_size()
    return int(total)


def theoretical_kv_cache_bytes(model, num_tokens, batch_size=1, bits_per_element=16):
    """KV = 2 * n_layers * n_kv_heads * head_dim * bytes_per_element * batch * seq_len.
    A reference DENSE fp16 formula (calculated, not measured) tied to a real
    observed cache length -- the common baseline every method's compression
    ratio can be measured against."""
    cfg = model.config
    n = int(cfg.num_hidden_layers)
    h = int(getattr(cfg, "num_key_value_heads", cfg.num_attention_heads))
    d = int(cfg.hidden_size // cfg.num_attention_heads)
    e = bits_per_element / 8.0
    return int(2 * n * h * d * e * batch_size * num_tokens)


def prune_past_key_values(past_key_values, keep_idx):
    """Optional helper for eviction-style methods: drop cache positions not
    in keep_idx. Quantization-style methods generally won't call this."""
    past_key_values = cache_to_legacy(past_key_values)
    keep_idx = keep_idx.to(device=device, dtype=torch.long)
    pruned = []
    for key, value in past_key_values:
        k = key.index_select(dim=2, index=keep_idx).contiguous()
        v = value.index_select(dim=2, index=keep_idx).contiguous()
        pruned.append((k, v))
    return tuple(pruned)


def print_kv_formula_constants():
    cfg = model.config
    n = int(cfg.num_hidden_layers)
    h = int(getattr(cfg, "num_key_value_heads", cfg.num_attention_heads))
    d = int(cfg.hidden_size // cfg.num_attention_heads)
    print("Dense KV-cache formula: KV = 2 * n_layers * n_kv_heads * head_dim * bytes_per_element * batch * seq_len")
    print("n layers:", n, "| h kv heads:", h, "| d head dim:", d)
    print("Reference dense fp16 KV at MAX_LENGTH:",
          theoretical_kv_cache_bytes(model, MAX_LENGTH, bits_per_element=16) / 1024**3, "GB")

In [ ]:
# Block 7 - COMPRESSION METHOD PLUG-IN: H2O (Heavy-Hitter Oracle)
#
# H2O evicts KV-cache tokens at every decoding step, keeping only:
#   - a fixed-size window of the most RECENT tokens, and
#   - the top-K "heavy hitter" tokens by cumulative attention score,
# discarding everything else. This is a genuine eviction -- the cache
# tensors are physically shrunk -- so measured_kv_cache_bytes below is a
# real measurement, not a formula (see calculated_kv_cache_bytes == None).
#
# REQUIRED_ATTN_IMPL must be "eager": H2O needs real attention weights for
# heavy-hitter scoring, and flash/sdpa don't expose them. This means H2O's
# TTFT/TBT numbers are NOT directly comparable to a method using flash/sdpa
# without a controlled confound check -- note this in any write-up rather
# than treating attention backend as free of effect.

METHOD_NAME = "h2o"
REQUIRED_ATTN_IMPL = "eager"

# H2O-specific budget knobs -- a property of the method itself, not part of
# the shared config, so these don't need to (and shouldn't) match anything
# in another method's notebook. Total cache budget = HEAVY_RATIO + RECENT_RATIO.
HEAVY_RATIO = 0.10
RECENT_RATIO = 0.10
MIN_CACHE_TOKENS = 16


def init_state():
    return {"past_key_values": None, "heavy_scores": None}


def _h2o_cache_budgets(total_len):
    total_budget = max(MIN_CACHE_TOKENS, int(total_len * (HEAVY_RATIO + RECENT_RATIO)))
    recent_budget = max(1, int(total_len * RECENT_RATIO))
    heavy_budget = max(0, total_budget - recent_budget)
    return total_budget, recent_budget, heavy_budget


def make_budgets(total_len):
    return _h2o_cache_budgets(total_len)


def _update_heavy_scores(heavy_scores, attention_scores, cache_len):
    if heavy_scores is None:
        heavy_scores = torch.zeros(cache_len, device=device, dtype=torch.float32)

    if heavy_scores.numel() < cache_len:
        pad = torch.zeros(cache_len - heavy_scores.numel(), device=device, dtype=torch.float32)
        heavy_scores = torch.cat([heavy_scores, pad], dim=0)
    elif heavy_scores.numel() > cache_len:
        heavy_scores = heavy_scores[:cache_len]

    attention_scores = attention_scores.to(device=device, dtype=torch.float32)
    if attention_scores.numel() == cache_len:
        heavy_scores += attention_scores
    else:
        n = min(heavy_scores.numel(), attention_scores.numel())
        heavy_scores[-n:] += attention_scores[-n:]
    return heavy_scores


def _apply_h2o_pruning(past_key_values, heavy_scores, current_cache_len, total_budget, recent_budget, heavy_budget):
    if current_cache_len <= total_budget:
        return past_key_values, heavy_scores, current_cache_len

    recent_count = min(recent_budget, current_cache_len)
    recent_start = current_cache_len - recent_count
    recent_idx = torch.arange(recent_start, current_cache_len, device=device, dtype=torch.long)

    old_end = recent_start
    k_heavy = min(heavy_budget, max(0, old_end))

    if k_heavy > 0 and heavy_scores is not None:
        heavy_idx = torch.topk(heavy_scores[:old_end], k=k_heavy).indices
    else:
        heavy_idx = torch.empty(0, device=device, dtype=torch.long)

    keep_idx = torch.cat([heavy_idx, recent_idx], dim=0).unique().sort().values

    past_key_values = prune_past_key_values(past_key_values, keep_idx)
    if heavy_scores is not None:
        heavy_scores = heavy_scores.index_select(dim=0, index=keep_idx)

    current_cache_len = get_cache_tokens(past_key_values)
    return past_key_values, heavy_scores, current_cache_len


@torch.inference_mode()
def forward_one_token(token_tensor, abs_pos, state, budgets):
    past_key_values = state["past_key_values"]
    heavy_scores = state["heavy_scores"]
    total_budget, recent_budget, heavy_budget = budgets

    cache_len_before = get_cache_tokens(past_key_values)
    attention_mask = torch.ones((1, cache_len_before + 1), dtype=torch.long, device=device)
    position_ids = torch.tensor([[abs_pos]], dtype=torch.long, device=device)

    outputs = model(
        input_ids=token_tensor,
        past_key_values=past_key_values,
        attention_mask=attention_mask,
        position_ids=position_ids,
        use_cache=True,
        output_attentions=True,
        return_dict=True,
    )

    past_key_values = cache_to_legacy(outputs.past_key_values)
    current_cache_len = get_cache_tokens(past_key_values)

    if outputs.attentions is not None and outputs.attentions[-1] is not None:
        att = outputs.attentions[-1].detach()
        att_score = att.float().mean(dim=(0, 1, 2))
    else:
        att_score = torch.zeros(current_cache_len, device=device, dtype=torch.float32)

    heavy_scores = _update_heavy_scores(heavy_scores, att_score, current_cache_len)

    past_key_values, heavy_scores, current_cache_len = _apply_h2o_pruning(
        past_key_values, heavy_scores, current_cache_len,
        total_budget, recent_budget, heavy_budget,
    )

    new_state = {"past_key_values": past_key_values, "heavy_scores": heavy_scores}
    return outputs.logits[:, -1, :], new_state


def get_state_cache_tokens(state):
    return get_cache_tokens(state["past_key_values"])


def measured_kv_cache_bytes(state):
    return dense_kv_cache_bytes(state["past_key_values"])


def calculated_kv_cache_bytes(state):
    return None  # H2O physically evicts tokens -- measured_kv_cache_bytes above is already the honest number

In [ ]:
# Block 8 - Generic scoring functions (perplexity / next-token accuracy)
# Reused unmodified across methods -- all method-specific behavior lives
# inside forward_one_token (Block 7).


def safe_exp(x):
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return float("nan")
    return math.exp(min(float(x), 50.0))


def _floats_match(a, b, tol=1e-4):
    try:
        return abs(float(a) - float(b)) < tol
    except (ValueError, TypeError):
        return False


@torch.inference_mode()
def score_input_ids(input_ids, score_start_index=1):
    input_ids = input_ids.to(device)
    seq_len = int(input_ids.shape[1])
    if seq_len < 2:
        return None

    budgets = make_budgets(seq_len)
    state = init_state()

    total_nll = 0.0
    scored_tokens = 0
    correct_tokens = 0
    max_measured_bytes = 0
    max_calculated_bytes = 0
    max_cache_tokens = 0

    for t in range(seq_len - 1):
        current_input = input_ids[:, t:t + 1]
        target = input_ids[:, t + 1]

        logits, state = forward_one_token(current_input, t, state, budgets)

        if (t + 1) >= score_start_index:
            log_probs = torch.log_softmax(logits.float(), dim=-1)
            total_nll += -log_probs[0, target].item()
            scored_tokens += 1
            pred = int(torch.argmax(logits, dim=-1)[0].item())
            correct_tokens += int(pred == int(target.item()))

        cache_tokens = get_state_cache_tokens(state)
        max_cache_tokens = max(max_cache_tokens, cache_tokens)
        max_measured_bytes = max(max_measured_bytes, measured_kv_cache_bytes(state))
        calc = calculated_kv_cache_bytes(state)
        if calc is not None:
            max_calculated_bytes = max(max_calculated_bytes, calc)

    avg_nll = total_nll / max(scored_tokens, 1)
    reported_bytes = max_calculated_bytes if max_calculated_bytes > 0 else max_measured_bytes

    return {
        "nll_sum": total_nll,
        "tokens": scored_tokens,
        "avg_nll": avg_nll,
        "perplexity": safe_exp(avg_nll),
        "correct_tokens": correct_tokens,
        "accuracy": correct_tokens / max(scored_tokens, 1),
        "max_cache_tokens": max_cache_tokens,
        "measured_kv_cache_gb": max_measured_bytes / 1024**3,
        "calculated_kv_cache_gb": (max_calculated_bytes / 1024**3) if max_calculated_bytes > 0 else None,
        "reported_kv_cache_gb": reported_bytes / 1024**3,
        "kv_cache_accounting": "calculated" if max_calculated_bytes > 0 else "measured",
        "reference_dense_kv_cache_gb": theoretical_kv_cache_bytes(model, max_cache_tokens, bits_per_element=16) / 1024**3,
    }


@torch.inference_mode()
def score_window(window_ids):
    input_ids = torch.tensor([window_ids], dtype=torch.long, device=device)
    if int(input_ids.shape[1]) < 8:
        return None
    return score_input_ids(input_ids, score_start_index=1)


@torch.inference_mode()
def score_prompt_target(prompt, target):
    prompt_ids = tokenizer(prompt, add_special_tokens=True)["input_ids"]
    target_text = " " + str(target).strip()
    target_ids = tokenizer(target_text, add_special_tokens=False)["input_ids"]

    max_prompt_len = max(1, MAX_LENGTH - len(target_ids))
    if len(prompt_ids) > max_prompt_len:
        prompt_ids = prompt_ids[-max_prompt_len:]

    full_ids = torch.tensor([prompt_ids + target_ids], dtype=torch.long, device=device)
    return score_input_ids(full_ids, score_start_index=len(prompt_ids))

In [ ]:
# Block 9 - Generic generation + latency measurement
# TTFT/TBT are always measured against whatever real prompt is passed in --
# for GSM8K that is the SAME fewshot-prefixed prompt used for accuracy (see
# Block 10), not a separate un-prefixed probe.


def _normalize_generated_answer(text, answer_type):
    text = str(text).strip()
    if answer_type in {"number", "number_string"}:
        return extract_gsm8k_final_answer(text)
    return text.lower().strip()


def _normalize_target_answer(text, answer_type):
    text = str(text).strip()
    if answer_type in {"number", "number_string"}:
        return normalize_number_text(text)
    return text.lower().strip()


@torch.inference_mode()
def generate_text(prompt, max_new_tokens):
    max_prompt_len = max(8, MAX_LENGTH - max_new_tokens)
    enc = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_prompt_len)
    prompt_ids = enc["input_ids"].to(device)
    prompt_len = int(prompt_ids.shape[1])

    budgets = make_budgets(prompt_len + max_new_tokens)
    state = init_state()
    generated_ids = []

    max_measured_bytes = 0
    max_calculated_bytes = 0
    max_cache_tokens = 0
    last_logits = None

    sync_if_cuda()
    generation_start = time.perf_counter()

    for t in range(prompt_len):
        token_tensor = prompt_ids[:, t:t + 1]
        last_logits, state = forward_one_token(token_tensor, t, state, budgets)
        cache_tokens = get_state_cache_tokens(state)
        max_cache_tokens = max(max_cache_tokens, cache_tokens)
        max_measured_bytes = max(max_measured_bytes, measured_kv_cache_bytes(state))
        calc = calculated_kv_cache_bytes(state)
        if calc is not None:
            max_calculated_bytes = max(max_calculated_bytes, calc)

    sync_if_cuda()
    ttft_sec = time.perf_counter() - generation_start

    step_times = []
    for step in range(max_new_tokens):
        next_id = int(torch.argmax(last_logits, dim=-1)[0].item())
        if next_id == tokenizer.eos_token_id:
            break
        generated_ids.append(next_id)

        token_tensor = torch.tensor([[next_id]], dtype=torch.long, device=device)
        abs_pos = prompt_len + step

        sync_if_cuda()
        step_start = time.perf_counter()
        last_logits, state = forward_one_token(token_tensor, abs_pos, state, budgets)
        sync_if_cuda()
        step_times.append(time.perf_counter() - step_start)

        cache_tokens = get_state_cache_tokens(state)
        max_cache_tokens = max(max_cache_tokens, cache_tokens)
        max_measured_bytes = max(max_measured_bytes, measured_kv_cache_bytes(state))
        calc = calculated_kv_cache_bytes(state)
        if calc is not None:
            max_calculated_bytes = max(max_calculated_bytes, calc)

    sync_if_cuda()
    total_generation_latency_sec = time.perf_counter() - generation_start
    tbt_sec = sum(step_times) / len(step_times) if step_times else 0.0
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    reported_bytes = max_calculated_bytes if max_calculated_bytes > 0 else max_measured_bytes

    return {
        "generated_text": generated_text,
        "generated_token_count": len(generated_ids),
        "prompt_tokens": prompt_len,
        "ttft_sec": ttft_sec,
        "tbt_sec": tbt_sec,
        "total_generation_latency_sec": total_generation_latency_sec,
        "max_cache_tokens": max_cache_tokens,
        "measured_kv_cache_gb": max_measured_bytes / 1024**3,
        "calculated_kv_cache_gb": (max_calculated_bytes / 1024**3) if max_calculated_bytes > 0 else None,
        "reported_kv_cache_gb": reported_bytes / 1024**3,
        "kv_cache_accounting": "calculated" if max_calculated_bytes > 0 else "measured",
        "reference_dense_kv_cache_gb": theoretical_kv_cache_bytes(model, max_cache_tokens, bits_per_element=16) / 1024**3,
    }


@torch.inference_mode()
def score_gsm8k_example(example):
    gen = generate_text(prompt=example["prompt"], max_new_tokens=int(example.get("max_new_tokens", GSM8K_MAX_NEW_TOKENS)))
    target_score = score_prompt_target(example["prompt"], example["target"])

    pred_norm = _normalize_generated_answer(gen["generated_text"], example.get("answer_type", "text"))
    target_norm = _normalize_target_answer(example["target"], example.get("answer_type", "text"))
    exact_match = int(target_norm != "" and _floats_match(pred_norm, target_norm))

    if target_score is None:
        avg_nll, ppl, nll_sum, target_tokens = float("nan"), float("nan"), 0.0, 0
        reported = gen["reported_kv_cache_gb"]
        measured = gen["measured_kv_cache_gb"]
        max_tokens = gen["max_cache_tokens"]
        accounting = gen["kv_cache_accounting"]
        dense_ref = gen["reference_dense_kv_cache_gb"]
    else:
        avg_nll = target_score["avg_nll"]
        ppl = target_score["perplexity"]
        nll_sum = target_score["nll_sum"]
        target_tokens = target_score["tokens"]
        reported = max(gen["reported_kv_cache_gb"], target_score["reported_kv_cache_gb"])
        measured = max(gen["measured_kv_cache_gb"], target_score["measured_kv_cache_gb"])
        max_tokens = max(gen["max_cache_tokens"], target_score["max_cache_tokens"])
        accounting = gen["kv_cache_accounting"]
        dense_ref = max(gen["reference_dense_kv_cache_gb"], target_score["reference_dense_kv_cache_gb"])

    return {
        "nll_sum": nll_sum,
        "tokens": gen["prompt_tokens"] + gen["generated_token_count"],
        "target_tokens": target_tokens,
        "avg_nll": avg_nll,
        "perplexity": ppl,
        "accuracy": exact_match,
        "pred_text": gen["generated_text"],
        "pred_norm": pred_norm,
        "target_norm": target_norm,
        "generated_token_count": gen["generated_token_count"],
        "prompt_tokens": gen["prompt_tokens"],
        "ttft_sec": gen["ttft_sec"],
        "tbt_sec": gen["tbt_sec"],
        "total_generation_latency_sec": gen["total_generation_latency_sec"],
        "max_cache_tokens": max_tokens,
        "measured_kv_cache_gb": measured,
        "reported_kv_cache_gb": reported,
        "kv_cache_accounting": accounting,
        "reference_dense_kv_cache_gb": dense_ref,
    }

In [ ]:
# Block 10 - Dataset-level evaluators
# Identical structure for every method -- what differs across methods lives
# entirely in Block 7.
#
# Two separate latency fields, so a per-token-loop method (this template)
# and a subprocess-based method (e.g. KVQuant's adapter) report genuinely
# comparable numbers instead of two different things under the same name:
#   - avg_total_latency_sec: TTFT + decode time for ONE prompt, averaged
#     across every natural prompt (or GSM8K example) -- comparable across
#     any method's notebook.
#   - dataset_wall_clock_sec: total wall-clock time to evaluate the WHOLE
#     dataset (every window's perplexity/accuracy pass + every prompt's
#     latency probe) -- NOT comparable to a subprocess method's per-run
#     timing unless both are measuring the exact same workload shape.


@torch.inference_mode()
def evaluate_lm_dataset(dataset_name, data):
    clear_memory()
    windows = data["windows"]
    natural_prompts = data["natural_prompts"]
    sample_records = []

    total_nll = 0.0
    total_tokens = 0
    total_correct_tokens = 0
    max_reference_dense_gb = 0.0
    max_measured_gb = 0.0
    max_calculated_gb = 0.0
    max_cache_tokens = 0

    sync_if_cuda()
    dataset_start = time.perf_counter()

    for sample_id, window_ids in enumerate(tqdm(windows, desc=f"{METHOD_NAME} | {dataset_name} | windows")):
        result = score_window(window_ids)
        if result is None:
            continue

        total_nll += result["nll_sum"]
        total_tokens += result["tokens"]
        total_correct_tokens += result["correct_tokens"]
        max_reference_dense_gb = max(max_reference_dense_gb, result["reference_dense_kv_cache_gb"])
        max_measured_gb = max(max_measured_gb, result["measured_kv_cache_gb"])
        if result["calculated_kv_cache_gb"] is not None:
            max_calculated_gb = max(max_calculated_gb, result["calculated_kv_cache_gb"])
        max_cache_tokens = max(max_cache_tokens, result["max_cache_tokens"])

        sample_records.append({
            "dataset": dataset_name, "dataset_type": "language_modeling", "method": METHOD_NAME,
            "sample_id": sample_id, "tokens": result["tokens"], "avg_nll": result["avg_nll"],
            "perplexity": result["perplexity"], "accuracy": result["accuracy"],
            "reported_kv_cache_gb": result["reported_kv_cache_gb"], "max_cache_tokens": result["max_cache_tokens"],
        })

    avg_nll = total_nll / max(total_tokens, 1)
    perplexity = safe_exp(avg_nll)
    accuracy = total_correct_tokens / max(total_tokens, 1)

    ttft_values, tbt_values, total_latency_values = [], [], []
    for prompt_text in tqdm(natural_prompts, desc=f"{METHOD_NAME} | {dataset_name} | latency probes"):
        probe = generate_text(prompt=prompt_text, max_new_tokens=TBT_GENERATED_TOKENS)
        ttft_values.append(probe["ttft_sec"])
        tbt_values.append(probe["tbt_sec"])
        total_latency_values.append(probe["total_generation_latency_sec"])
        max_reference_dense_gb = max(max_reference_dense_gb, probe["reference_dense_kv_cache_gb"])
        max_measured_gb = max(max_measured_gb, probe["measured_kv_cache_gb"])
        if probe["calculated_kv_cache_gb"] is not None:
            max_calculated_gb = max(max_calculated_gb, probe["calculated_kv_cache_gb"])
        max_cache_tokens = max(max_cache_tokens, probe["max_cache_tokens"])

    ttft_sec = sum(ttft_values) / len(ttft_values) if ttft_values else float("nan")
    tbt_sec = sum(tbt_values) / len(tbt_values) if tbt_values else float("nan")
    avg_total_latency_sec = sum(total_latency_values) / len(total_latency_values) if total_latency_values else float("nan")

    sync_if_cuda()
    dataset_wall_clock_sec = time.perf_counter() - dataset_start
    reported_kv_gb = max_calculated_gb if max_calculated_gb > 0 else max_measured_gb
    accounting = "calculated" if max_calculated_gb > 0 else "measured"

    row = {
        "dataset": dataset_name, "dataset_type": "language_modeling", "method": METHOD_NAME,
        "samples": len(sample_records), "tokens": total_tokens,
        "perplexity": perplexity, "accuracy": accuracy, "accuracy_type": "next_token_accuracy",
        "reference_dense_kv_cache_gb": max_reference_dense_gb,
        "measured_kv_cache_gb": max_measured_gb,
        "calculated_kv_cache_gb": max_calculated_gb if max_calculated_gb > 0 else None,
        "reported_kv_cache_gb": reported_kv_gb, "kv_cache_accounting": accounting,
        "max_cache_tokens": max_cache_tokens,
        "ttft_sec": ttft_sec, "tbt_sec": tbt_sec, "ttft_tbt_n_prompts": len(natural_prompts),
        "avg_total_latency_sec": avg_total_latency_sec,
        "dataset_wall_clock_sec": dataset_wall_clock_sec,
        "ms_per_token": 1000.0 * dataset_wall_clock_sec / max(total_tokens, 1),
        "tokens_per_sec": total_tokens / max(dataset_wall_clock_sec, 1e-9),
    }
    return row, sample_records


@torch.inference_mode()
def evaluate_generation_dataset(dataset_name, examples):
    clear_memory()
    sample_records = []

    total_nll = 0.0
    total_target_tokens = 0
    total_runtime_tokens = 0
    total_correct = 0
    max_reference_dense_gb = 0.0
    max_measured_gb = 0.0
    max_calculated_gb = 0.0
    max_cache_tokens = 0
    ttft_values, tbt_values, total_latency_values = [], [], []

    sync_if_cuda()
    dataset_start = time.perf_counter()

    for sample_id, example in enumerate(tqdm(examples, desc=f"{METHOD_NAME} | {dataset_name}")):
        result = score_gsm8k_example(example)
        if result is None:
            continue

        total_nll += result["nll_sum"]
        total_target_tokens += result["target_tokens"]
        total_runtime_tokens += result["tokens"]
        total_correct += result["accuracy"]
        ttft_values.append(result["ttft_sec"])
        tbt_values.append(result["tbt_sec"])
        total_latency_values.append(result["total_generation_latency_sec"])
        max_reference_dense_gb = max(max_reference_dense_gb, result["reference_dense_kv_cache_gb"])
        max_measured_gb = max(max_measured_gb, result["measured_kv_cache_gb"])
        if result["kv_cache_accounting"] == "calculated":
            max_calculated_gb = max(max_calculated_gb, result["reported_kv_cache_gb"])
        max_cache_tokens = max(max_cache_tokens, result["max_cache_tokens"])

        sample_records.append({
            "dataset": dataset_name, "dataset_type": "generation", "method": METHOD_NAME,
            "sample_id": sample_id, "accuracy": result["accuracy"], "target_norm": result["target_norm"],
            "pred_norm": result["pred_norm"], "pred_text": result["pred_text"], "tokens": result["tokens"],
            "target_tokens": result["target_tokens"], "avg_nll": result["avg_nll"], "perplexity": result["perplexity"],
            "generated_token_count": result["generated_token_count"], "prompt_tokens": result["prompt_tokens"],
            "ttft_sec": result["ttft_sec"], "tbt_sec": result["tbt_sec"],
            "reported_kv_cache_gb": result["reported_kv_cache_gb"], "max_cache_tokens": result["max_cache_tokens"],
        })

    sync_if_cuda()
    dataset_wall_clock_sec = time.perf_counter() - dataset_start
    avg_nll = total_nll / max(total_target_tokens, 1)
    ttft_sec = sum(ttft_values) / len(ttft_values) if ttft_values else float("nan")
    tbt_sec = sum(tbt_values) / len(tbt_values) if tbt_values else float("nan")
    avg_total_latency_sec = sum(total_latency_values) / len(total_latency_values) if total_latency_values else float("nan")
    reported_kv_gb = max_calculated_gb if max_calculated_gb > 0 else max_measured_gb
    accounting = "calculated" if max_calculated_gb > 0 else "measured"

    row = {
        "dataset": dataset_name, "dataset_type": "generation", "method": METHOD_NAME,
        "samples": len(sample_records), "tokens": total_runtime_tokens,
        "perplexity": safe_exp(avg_nll), "accuracy": total_correct / max(len(sample_records), 1),
        "accuracy_type": "gsm8k_numeric_exact_match",
        "reference_dense_kv_cache_gb": max_reference_dense_gb,
        "measured_kv_cache_gb": max_measured_gb,
        "calculated_kv_cache_gb": max_calculated_gb if max_calculated_gb > 0 else None,
        "reported_kv_cache_gb": reported_kv_gb, "kv_cache_accounting": accounting,
        "max_cache_tokens": max_cache_tokens,
        "ttft_sec": ttft_sec, "tbt_sec": tbt_sec, "ttft_tbt_n_prompts": len(sample_records),
        "avg_total_latency_sec": avg_total_latency_sec,
        "dataset_wall_clock_sec": dataset_wall_clock_sec,
        "ms_per_token": 1000.0 * dataset_wall_clock_sec / max(total_runtime_tokens, 1),
        "tokens_per_sec": total_runtime_tokens / max(dataset_wall_clock_sec, 1e-9),
    }
    return row, sample_records

In [ ]:
# Block 11 - Run this method on PTB, WikiText-103, and GSM8K

summary_rows = []
sample_rows = []

load_model(REQUIRED_ATTN_IMPL)
print_kv_formula_constants()

print()
print("=" * 80)
print(f"Running {METHOD_NAME}, active_attn_impl={active_attn_impl}")
print("=" * 80)
print("LM datasets:", {k: len(v["windows"]) for k, v in lm_datasets.items()})
print("Generation datasets:", {k: len(v) for k, v in generation_datasets.items()})

for dataset_name, data in lm_datasets.items():
    if not data["windows"]:
        print(f"Skipping {dataset_name}: no LM windows.")
        continue
    print()
    print("-" * 80)
    print("Evaluating LM dataset:", dataset_name)
    print("-" * 80)
    row, records = evaluate_lm_dataset(dataset_name, data)
    summary_rows.append(row)
    sample_rows.extend(records)
    print(row)

if "GSM8K" in generation_datasets and len(generation_datasets["GSM8K"]) > 0:
    print()
    print("-" * 80)
    print("Evaluating generation dataset: GSM8K")
    print("-" * 80)
    row, records = evaluate_generation_dataset("GSM8K", generation_datasets["GSM8K"])
    summary_rows.append(row)
    sample_rows.extend(records)
    print(row)
else:
    print("ERROR: GSM8K not available.")

results_df = pd.DataFrame(summary_rows)
samples_df = pd.DataFrame(sample_rows)

print()
print("Final results_df datasets:", results_df["dataset"].tolist() if not results_df.empty else [])
display(results_df)

In [ ]:
# Block 12 - Build the baseline table

DATASET_ORDER = ["PTB", "WikiText-103", "GSM8K"]
METRIC_ROWS = [
    "Perplexity", "Accuracy", "Reference Dense Memory (GB)", "Reported Memory (GB)",
    "TTFT (s)", "TBT (s)", "Avg Total Latency (s)", "Dataset Wall-Clock Time (s)",
]


def _get_metric_value(df, dataset, metric):
    rows = df[df["dataset"] == dataset]
    if rows.empty:
        return "N/A"
    r = rows.iloc[0]
    mapping = {
        "Perplexity": "perplexity", "Accuracy": "accuracy",
        "Reference Dense Memory (GB)": "reference_dense_kv_cache_gb",
        "Reported Memory (GB)": "reported_kv_cache_gb",
        "TTFT (s)": "ttft_sec", "TBT (s)": "tbt_sec",
        "Avg Total Latency (s)": "avg_total_latency_sec",
        "Dataset Wall-Clock Time (s)": "dataset_wall_clock_sec",
    }
    v = r.get(mapping[metric], float("nan"))
    if pd.isna(v):
        return "N/A"
    if metric == "Accuracy":
        return f"{float(v):.4f}"
    if metric in {"Reference Dense Memory (GB)", "Reported Memory (GB)"}:
        return f"{float(v):.6f}"
    return f"{float(v):.4f}"


table_rows = [{"Method": METHOD_NAME, **{ds: "" for ds in DATASET_ORDER}}]
for metric in METRIC_ROWS:
    table_rows.append({"Method": metric, **{ds: _get_metric_value(results_df, ds, metric) for ds in DATASET_ORDER}})

baseline_table_df = pd.DataFrame(table_rows, columns=["Method"] + DATASET_ORDER)

accounting = results_df["kv_cache_accounting"].iloc[0] if not results_df.empty else "n/a"
print(f"{METHOD_NAME} baseline table (memory accounting for this method: {accounting}):")
print("Avg Total Latency = TTFT + decode time for ONE prompt, averaged across prompts (comparable across methods).")
print("Dataset Wall-Clock Time = total time to evaluate the WHOLE dataset (NOT comparable to a different workload shape).")
display(baseline_table_df)
display(results_df)

In [ ]:
# Block 13 - Save results, namespaced by method so every method's CSVs can
# later be concatenated (they all share the same "method"/"dataset" columns)
# into one cross-method comparison table.

summary_path = f"/content/{METHOD_NAME}_baseline_summary.csv"
samples_path = f"/content/{METHOD_NAME}_baseline_samples.csv"
table_path = f"/content/{METHOD_NAME}_baseline_table.csv"

results_df.to_csv(summary_path, index=False)
samples_df.to_csv(samples_path, index=False)
baseline_table_df.to_csv(table_path, index=False)

print("Saved summary:", summary_path)
print("Saved samples:", samples_path)
print("Saved table:", table_path)

In [ ]:
# Debug cell -- inspect GSM8K generated answers if accuracy looks off

if "samples_df" in globals() and not samples_df.empty:
    gsm8k_debug = samples_df[samples_df["dataset"] == "GSM8K"].copy()
    if not gsm8k_debug.empty:
        cols = [c for c in ["sample_id", "accuracy", "target_norm", "pred_norm", "generated_token_count", "pred_text"] if c in gsm8k_debug.columns]
        pd.set_option("display.max_colwidth", 1000)
        display(gsm8k_debug[cols])
        print()
        print("GSM8K exact-match accuracy:", int(gsm8k_debug["accuracy"].sum()), "/", len(gsm8k_debug), "=", float(gsm8k_debug["accuracy"].mean()))
        empty_pred = gsm8k_debug[gsm8k_debug["pred_norm"].astype(str).str.len() == 0]
        print()
        print("Cases where the parser found no prediction:")
        display(empty_pred[cols])
    else:
        print("No GSM8K sample rows found.")
else:
    print("Run Block 11 first so samples_df exists.")

In [ ]:
# Block 14 - Simple plots for this method alone (compare across methods by
# loading multiple *_baseline_summary.csv files together afterward)

if not results_df.empty:
    dataset_order = ["PTB", "WikiText-103", "GSM8K"]
    plot_df = results_df.set_index("dataset").reindex(dataset_order).reset_index()

    def bar_values(series):
        return [0 if pd.isna(v) else float(v) for v in series.tolist()]

    plt.figure(figsize=(7, 4))
    plt.bar(plot_df["dataset"].astype(str), bar_values(plot_df["perplexity"]))
    plt.ylabel("Perplexity"); plt.title(f"{METHOD_NAME} Perplexity"); plt.xticks(rotation=30); plt.grid(True); plt.show()

    plt.figure(figsize=(7, 4))
    plt.bar(plot_df["dataset"].astype(str), bar_values(plot_df["accuracy"]))
    plt.ylabel("Accuracy"); plt.ylim(0, 1.05); plt.title(f"{METHOD_NAME} Accuracy"); plt.xticks(rotation=30); plt.grid(True); plt.show()

    plt.figure(figsize=(7, 4))
    plt.plot(plot_df["dataset"].astype(str), plot_df["reference_dense_kv_cache_gb"], marker="o", label="reference dense (fp16)")
    plt.plot(plot_df["dataset"].astype(str), plot_df["reported_kv_cache_gb"], marker="o", label=f"reported ({results_df['kv_cache_accounting'].iloc[0]})")
    plt.ylabel("KV-cache memory (GB)"); plt.title(f"{METHOD_NAME} KV-cache memory"); plt.xticks(rotation=30); plt.legend(); plt.grid(True); plt.show()

    plt.figure(figsize=(7, 4))
    plt.plot(plot_df["dataset"].astype(str), plot_df["ttft_sec"], marker="o", label="TTFT")
    plt.plot(plot_df["dataset"].astype(str), plot_df["tbt_sec"], marker="o", label="TBT")
    plt.ylabel("Seconds"); plt.title(f"{METHOD_NAME} TTFT / TBT"); plt.xticks(rotation=30); plt.legend(); plt.grid(True); plt.show()

    plt.figure(figsize=(7, 4))
    plt.bar(plot_df["dataset"].astype(str), bar_values(plot_df["dataset_wall_clock_sec"]))
    plt.ylabel("Seconds"); plt.title(f"{METHOD_NAME} dataset wall-clock time"); plt.xticks(rotation=30); plt.grid(True); plt.show()
else:
    print("results_df is empty. Run Block 11 first.")

## Notes for comparing to KVQuant

- This notebook's `results_df`/CSV schema is identical to
  `KVQuant_Implementation_v2.ipynb`'s — both share every Block-1-through-6
  and Block-8-through-14 cell verbatim, so their `*_baseline_summary.csv`
  files concatenate directly by `dataset`/`method` for a cross-method table.
- **Memory accounting**: this notebook's `reported_kv_cache_gb` is
  `measured` — H2O's eviction physically shrinks the cache, so this is a
  genuine live measurement. KVQuant's equivalent number is `calculated` — a
  formula applied to real tensor shapes, since its simulated-quantization
  pipeline never physically shrinks or re-encodes anything. Both numbers
  are tagged with `kv_cache_accounting` explicitly so they're never
  compared as if they were the same kind of quantity.
- **Attention backend**: forced to `eager` here (structural requirement,
  not a confound to "fix") — KVQuant auto-selects flash/sdpa. Run KVQuant
  once with `FORCE_ATTN_IMPL` set to match if you want an isolated
  attention-backend confound check.
- **Calibration**: H2O has none; KVQuant calibrates once on WikiText2 and
  is evaluated cross-domain. State this as an interpretive caveat in your
  write-up rather than trying to equalize it.
- Before trusting a cross-method comparison: confirm both notebooks ran on
  the same actual Colab GPU type — neither notebook pins that for you.